In [ ]:
import pandas as pd
from google.colab import drive
import re
import random
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import nltk

In [ ]:
# Download the stop words from NLTK
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# Get the German stopwords list
from nltk.corpus import stopwords
german_stopwords = stopwords.words('german')

In [ ]:
!pip install transformers

In [ ]:
!pip install --upgrade huggingface-hub

In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 MB 29.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.5 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.7-cp311-cp311-linux_x86_64.whl size=4601133 sha256=097ff13c5eab302c8484aa609a41624bc5012881963d739d65357ab7a7d0d8d6
  Stored in directory: /root/.cache/pip/wheels/eb/82/79/ac77fcd49324b75ae6aa18e63a87cf9da4371a57e2cdc8dc03
Successfully built llama-cpp-python


In [ ]:
## INSTALLING HUGGINGFACE
!pip install huggingface-hub==0.17.1 -q

## INSTALLING llama-cpp-python
# GPU llama-cpp-python; Starting from version llama-cpp-python==0.1.79, it supports GGUF
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --upgrade --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.8/294.8 kB 20.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.48.3 requires huggingface-hub<1.0,>=0.24.0, but you have huggingface-hub 0.17.1 which is incompatible.
sentence-transformers 3.4.1 requires huggingface-hub>=0.20.0, but you have huggingface-hub 0.17.1 which is incompatible.
accelerate 1.3.0 requires huggingface-hub>=0.21.0, but you have huggingface-hub 0.17.1 which is incompatible.
peft 0.14.0 requires huggingface-hub>=0.25.0, but you have huggingface-hub 0.17.1 which is incompatible.
diffusers 0.32.2 requires huggingface-hub>=0.23.2, but you have huggingface-hub 0.17.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 MB 126.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installi

In [ ]:
# Connect to drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Read file with flood articles filtered by Jan

flood_articles  = pd.read_parquet("/content/drive/MyDrive/0. Postdoc/flood-events-germany/flood_articles.parquet")
flood_articles = flood_articles[flood_articles['year'] >= 2021]
flood_articles['date'] = pd.to_datetime(flood_articles[['year', 'month', 'day']])

<ipython-input-8-0d7aacf861a8>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flood_articles['date'] = pd.to_datetime(flood_articles[['year', 'month', 'day']])


In [ ]:
flood_articles.shape

(452527, 11)

In [ ]:
flood_articles.head()

,id_doc,title,text,source_id,name_source,date,year,month,day,subtitle,ressort
4786,PMG41-BKU20210112-1066703,13 Tote in Indonesien,Jakarta - Nach zwei Erdrutschen am Wochenende ...,BKU,Berliner Kurier,2021-01-12,2021.0,1.0,12.0,None,PANORAMA
4788,PMG41-BKU20210110-1063831,Alarm! Flick schustert die Bayern nicht dicht,Mönchengladbach - 14 Monate nach der Entlassun...,BKU,Berliner Kurier,2021-01-10,2021.0,1.0,10.0,Flut von Gegentoren sorgt für erste Krise unte...,SPORT
4789,PMG41-BKU20210111-1065255,Warum Testzentren in Berlin boomen,In der Hauptstadt herrscht Gründerzeitstimmung...,BKU,Berliner Kurier,2021-01-11,2021.0,1.0,11.0,"Amtsärzte und Politiker sprechen von ""Wildwuch...",HINTERGRUND
4790,PMG41-BKU20210208-1376318,Verdreckte Spree,Berlin -Stille Wasser sind tief - im Fall der ...,BKU,Berliner Kurier,2021-02-08,2021.0,2.0,8.0,Auf dem Grund des Flusses liegen große Mengen ...,HINTERGRUND
4791,PMG41-BKU20210126-1341418,Plötzlich ein Star! Wie der Hype um diese Hand...,Washington - Jennifer Ellis hat den linken US-...,BKU,Berliner Kurier,2021-01-26,2021.0,1.0,26.0,"Jennifer Ellis hat die Fäustlinge gestrickt, m...",PANORAMA


Function to remove duplicates

In [ ]:
def deduplicate_texts(df, text_column='text', similarity_threshold=0.85, chunk_size=1000):
    # Initialize TF-IDF vectorizer
    vectorizer = TfidfVectorizer(
        stop_words=german_stopwords,
        max_features=10000,
        ngram_range=(1, 2)
    )

    # Clean texts
    print("Preprocessing texts...")
    texts = df[text_column].fillna('').str.lower()

    # Get total number of texts
    n_texts = len(texts)

    # Vectorize the entire dataset at once
    tfidf_matrix = vectorizer.fit_transform(texts)

    # Calculate the full cosine similarity matrix for the entire dataset
    print("Calculating cosine similarities...")
    similarity_matrix = cosine_similarity(tfidf_matrix)

    # Initialize indices to keep
    indices_to_keep = set(range(n_texts))

    # Process in chunks to manage memory
    for i in tqdm(range(0, n_texts, chunk_size), desc="Processing chunks"):
        # Get current chunk indices
        chunk_indices = list(range(i, min(i + chunk_size, n_texts)))

        # Skip if all indices in chunk have been marked for removal
        if not any(idx in indices_to_keep for idx in chunk_indices):
            continue

        # Create a sub-matrix for the chunk
        chunk_similarity_matrix = similarity_matrix[chunk_indices, :][:, chunk_indices]

        # Find duplicates within the chunk
        for idx1 in range(len(chunk_indices)):
            if chunk_indices[idx1] not in indices_to_keep:
                continue

            for idx2 in range(idx1 + 1, len(chunk_indices)):
                if chunk_indices[idx2] not in indices_to_keep:
                    continue

                if chunk_similarity_matrix[idx1, idx2] > similarity_threshold:
                    # Keep the longer text
                    len1 = len(texts.iloc[chunk_indices[idx1]])
                    len2 = len(texts.iloc[chunk_indices[idx2]])
                    if len1 >= len2:
                        indices_to_keep.remove(chunk_indices[idx2])
                    else:
                        indices_to_keep.remove(chunk_indices[idx1])
                        break

    # Return deduplicated DataFrame
    deduplicated_df = df.iloc[list(indices_to_keep)].copy()
    print(f"Removed {len(df) - len(deduplicated_df)} duplicate texts")

    return deduplicated_df

Function to filter with new keywords

In [ ]:
def filter_articles_by_keywords(df, keywords):
    """
    Filters a DataFrame based on the presence of keywords in the 'title' and 'subtitle' columns.

    Parameters:
    df (pd.DataFrame): The input DataFrame containing 'title' and 'subtitle' columns.
    keywords (list): A list of keywords to search for in the 'title' and 'subtitle' columns.

    Returns:
    pd.DataFrame: A filtered DataFrame containing only rows where 'title' or 'subtitle' contains at least one keyword.
    """
    pattern = '|'.join(map(re.escape, keywords))  # Create regex pattern with OR condition
    # Filter rows where either 'title' or 'subtitle' contains a keyword
    return df[df['title'].str.contains(pattern, case=False, na=False, regex=True) |
              df['subtitle'].str.contains(pattern, case=False, na=False, regex=True)]

In [ ]:
keywords = ["tot", "Todesfall", "Todesfälle", "ums Leben", "gestorben", "Todesopfer",
            "mehrere Todesopfer", "geforderte Todesopfer", "tödliche Hochwasser",
            "ums Leben gekommen", "kamen ums Leben", "Menschen starben",
            "Menschenleben gekostet", "Menschenleben", "ertrunken", "getötet",
            "Leiche", "verschüttet", "in Autos ertrunken"]

filtered_df = filter_articles_by_keywords(flood_articles, keywords)

In [ ]:
filtered_df.shape

(9427, 11)

In [ ]:
deduplicated_df = deduplicate_texts(
    df=filtered_df,
    similarity_threshold=0.85,
    chunk_size=1000)

Preprocessing texts...
Calculating cosine similarities...


Processing chunks: 100%|██████████| 10/10 [00:00<00:00, 11.87it/s]

Removed 3433 duplicate texts


In [ ]:
# Save the deduplicated DataFrame to a Parquet file
deduplicated_df.to_parquet('/content/drive/MyDrive/0. Postdoc/flood-events-germany/deduplicated_data.parquet', engine='pyarrow')

In [ ]:
deduplicated_df = pd.read_parquet('/content/drive/MyDrive/0. Postdoc/flood-events-germany/deduplicated_data.parquet', engine='pyarrow')

In [ ]:
deduplicated_df.shape

(5994, 11)

In [ ]:
deduplicated_df.groupby('year').count()

,id_doc,title,text,source_id,name_source,date,month,day,subtitle,ressort
year,,,,,,,,,,
2021.0,3139,3139,3139,3139,3139,3139,3139,3139,1775,2846
2022.0,2855,2855,2855,2855,2855,2855,2855,2855,1413,2615


Using an LLM to ask if the event happened in Germany

Select LLM

In [ ]:
# @title Select Large Language Model
selected_llm = 'Mistral-7B-OpenOrca' # @param ["Mistral-7B", "Mistral-7B-OpenOrca", "Llama-2-13B-Chat"]

model_dic = {"Mistral-7B":{"HF_REPO_NAME":"TheBloke/Mistral-7B-Instruct-v0.1-GGUF","HF_MODEL_NAME":"mistral-7b-instruct-v0.1.Q4_K_M.gguf"},
           "Mistral-7B-OpenOrca":{"HF_REPO_NAME":"TheBloke/Mistral-7B-OpenOrca-GGUF","HF_MODEL_NAME":"mistral-7b-openorca.Q5_K_M.gguf"},
             "Llama-2-13B-Chat":{"HF_REPO_NAME":"TheBloke/Llama-2-13B-chat-GGUF","HF_MODEL_NAME":"llama-2-13b-chat.Q4_K_S.gguf"}
             }

In [ ]:
import os

from huggingface_hub import hf_hub_download


HF_REPO_NAME = model_dic[selected_llm]['HF_REPO_NAME']
HF_MODEL_NAME = model_dic[selected_llm]['HF_MODEL_NAME']
LOCAL_DIR_NAME = "models"

os.makedirs(LOCAL_DIR_NAME, exist_ok=True)
model_path = hf_hub_download(
    repo_id=HF_REPO_NAME, filename=HF_MODEL_NAME, local_dir=LOCAL_DIR_NAME
)

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_threads=2, # CPU cores
    n_batch=512, # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    n_gpu_layers=30, # The max for this model is 30 in a T4, If you use llama 2 70B, you'll need to put fewer layers on the GPU
    n_ctx=32000#4096, # Context window
)

llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from models/mistral-7b-openorca.Q5_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = open-orca_mistral-7b-openorca
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attenti

In [ ]:
def create_prompt(title, text):
    prompt = f"""
    You are tasked with determining if the given text refers to a flood event that occurred in Germany. Carefully read the text and answer with "1" if the event is indeed a flood in Germany, and "0" if it does not refer to a flood event in Germany. Be sure to consider the details about location and event type in your response.

    Title: {title}
    Text:
    {text}
    Answer:
    """
    return prompt

In [ ]:
# Define the output file to save progress
output_file = "/content/drive/MyDrive/0. Postdoc/flood-events-germany/deduplicated_results.csv"

# Load existing progress if the file exists
try:
    deduplicated_df = pd.read_csv(output_file)
except FileNotFoundError:
    pass  # If the file does not exist, proceed as usual

# Ensure 'llm_check' column exists
if 'llm_check' not in deduplicated_df.columns:
    deduplicated_df['llm_check'] = None

for index, row in deduplicated_df.iterrows():  # Iterate through the rows of the DataFrame
    if pd.notna(row['llm_check']):
        continue  # Skip already processed rows to avoid redoing work

    title = row['title']
    text = row['text'][:500]  # Cut the 'text' down to the first 500 characters
    prompt = create_prompt(title, text)  # Create the prompt with the truncated text
    response = llm(prompt, stream=True, stop=["\n\n"], temperature=0, max_tokens=20)  # Get response from the LLM
    print(prompt)

    result = ""
    for output in response:
        result += output['choices'][0]['text']  # Extract and accumulate the text from the response

    result = result.strip()  # Clean up the result
    print(result)

    # Store the response in the DataFrame
    deduplicated_df.at[index, 'llm_check'] = result

    # Save progress to CSV after each iteration
    deduplicated_df.to_csv(output_file, index=False)


    You are tasked with determining if the given text refers to a flood event that occurred in Germany. Carefully read the text and answer with "1" if the event is indeed a flood in Germany, and "0" if it does not refer to a flood event in Germany. Be sure to consider the details about location and event type in your response.

    Title: Viele Tote bei heftigen Unwettern in Europa
    Text:
    Paris/Rom/Wien Bei heftigen Unwettern sind im Mittelmeerraum und in Österreich mindestens 13 Menschen getötet worden. Allein auf der französischen Mittelmeerinsel Korsika kamen am Donnerstag sechs Menschen ums Leben, wie die Behörden am Freitag mitteilten.Über Korsika waren Böen mit einer Geschwindigkeit von mehr als 200 Kilometern pro Stunde gezogen. 45.000 Haushalte waren zeitweise ohne Strom. Auch in anderen Teilen Frankreichs wie in Marseille gab es Unwetter und überflutete Straßen.Die schwe
    Answer:
    


llama_perf_context_print:        load time =   31934.62 ms
llama_perf_context_print: prompt eval time =   31934.19 ms /   265 tokens (  120.51 ms per token,     8.30 tokens per second)
llama_perf_context_print:        eval time =     377.20 ms /     1 runs   (  377.20 ms per token,     2.65 tokens per second)
llama_perf_context_print:       total time =   32312.90 ms /   266 tokens


0

    You are tasked with determining if the given text refers to a flood event that occurred in Germany. Carefully read the text and answer with "1" if the event is indeed a flood in Germany, and "0" if it does not refer to a flood event in Germany. Be sure to consider the details about location and event type in your response.

    Title: 180 Tote bei Überflutungen
    Text:
    Kabul Durch schwere Überschwemmungen sind in Afghanistan innerhalb eines Monats mehr als 180 Menschen ums Leben gekommen. Etwa 3000 Häuser seien zerstört worden, sagte am Donnerstag ein Sprecher der Taliban-Regierung in Kabul. " Wenn die Überschwemmungen sich verstärken, hat das Islamische Emirat Afghanistan nicht so viele Ressourcen, um den Auswirkungen zu begegnen" , sagte Regierungssprecher Sabihullah Mudschahid. afp
    Answer:
    


Llama.generate: 81 prefix-match hit, remaining 160 prompt tokens to eval
llama_perf_context_print:        load time =   31934.62 ms
llama_perf_context_print: prompt eval time =   20000.47 ms /   160 tokens (  125.00 ms per token,     8.00 tokens per second)
llama_perf_context_print:        eval time =     383.15 ms /     1 runs   (  383.15 ms per token,     2.61 tokens per second)
llama_perf_context_print:       total time =   20385.20 ms /   161 tokens


0

    You are tasked with determining if the given text refers to a flood event that occurred in Germany. Carefully read the text and answer with "1" if the event is indeed a flood in Germany, and "0" if it does not refer to a flood event in Germany. Be sure to consider the details about location and event type in your response.

    Title: Pakistan beklagtfast tausend Tote
    Text:
    Islamabad Pakistan hat aufgrund der anhaltenden Regenfälle und daraus resultierenden Fluten den Notstand ausgerufen. Zudem hat das Land am Donnerstag um internationale Hilfe zur Bewältigung der Katastrophe gebeten. Die Todeszahlen steigen indes weiter: Nach neuesten Zahlen der Nationalen Katastrophenbehörde haben bereits 937 Menschen ihr Leben verloren. Außerdem seien 33 Millionen Menschen aufgrund der Fluten ohne feste Bleibe. Das seien 15 Prozent der pakistanischen Bevölkerung. " Wir richten d
    Answer:
    


...

0

    You are tasked with determining if the given text refers to a flood event that occurred in Germany. Carefully read the text and answer with "1" if the event is indeed a flood in Germany, and "0" if it does not refer to a flood event in Germany. Be sure to consider the details about location and event type in your response.

    Title: Gleiche Diagnose
    Text:
    Nein, Affenpockenviren werden keine nächste Pandemie auslösen. Der Erreger ist bekannt, nur sein Name ist noch immer irreführend: Das Virus ist in Nagetieren heimisch, Affen sind wie Menschen sogenannte Fehlwirte. Das Virus wird beim Geschlechtsverkehr oder per Tropfen bei körperlicher Nähe übertragen, aber nicht per Aerosol. Es gibt Impfstoffe, Groß­bri­tannien setzt sie bereits zu Ringimpfungen ein. Und es gibt wirksame antivirale Medikamente.Aber der erste Angstreflex angesichts der ­Meldunge
    Answer:
    


Llama.generate: 81 prefix-match hit, remaining 176 prompt tokens to eval
llama_perf_context_print:        load time =   31934.62 ms
llama_perf_context_print: prompt eval time =   21466.77 ms /   176 tokens (  121.97 ms per token,     8.20 tokens per second)
llama_perf_context_print:        eval time =     371.36 ms /     1 runs   (  371.36 ms per token,     2.69 tokens per second)
llama_perf_context_print:       total time =   21839.84 ms /   177 tokens


0

    You are tasked with determining if the given text refers to a flood event that occurred in Germany. Carefully read the text and answer with "1" if the event is indeed a flood in Germany, and "0" if it does not refer to a flood event in Germany. Be sure to consider the details about location and event type in your response.

    Title: »Alles noch in der Schwebe«
    Text:
    ******Das Schachspiel um die Macht beginnt. Vielleicht wird es das ganze Jahr andauern. Eine  Partie, die mit Tempo, Klugheit und zum Teil auch mit Raffinement durchgespielt werden wird.«  Das schrieb Joseph Goebbels unter dem Datum des 7. Januar 1932. Die viel zitierten Sätze  finden sich freilich nicht im Originaltagebuch, sondern in seinem Buch  Vom Kaiserhof zur  Reichskanzlei,  das 1934 im Franz Eher-Verlag erschien. Mit dieser stark bearbeiteten  und stilisierten Version seiner ursprünglic
    Answer:
    


Llama.generate: 81 prefix-match hit, remaining 180 prompt tokens to eval
llama_perf_context_print:        load time =   31934.62 ms
llama_perf_context_print: prompt eval time =   21913.84 ms /   180 tokens (  121.74 ms per token,     8.21 tokens per second)
llama_perf_context_print:        eval time =     377.81 ms /     1 runs   (  377.81 ms per token,     2.65 tokens per second)
llama_perf_context_print:       total time =   22293.22 ms /   181 tokens


0


In [ ]:
deduplicated_df[deduplicated_df.llm_check == 1].title

,title


In [ ]:
def create_prompt_check_fatalities(title, text):
    prompt = f"""
    You are tasked with determining if the given text explicitly mentions **fatalities** caused by a **flood** in **Germany**. Carefully read the text and answer with "1" if the **explicitly mentions deaths** due to a **flood in Germany**, and "0" if it does **not** mention such fatalities. Be sure to consider the details about location and event type in your response.

    Title: {title}
    Text:
    {text}
    Answer:
    """
    return prompt

In [ ]:
# Define the output file to save progress
output_file = "/content/drive/MyDrive/0. Postdoc/flood-events-germany/df_classified_articles_relevant_fatalities_check.csv"

# Load existing progress if the file exists
try:
    df_classified_articles_relevant = pd.read_csv(output_file)
except FileNotFoundError:
    pass  # If the file does not exist, proceed as usual

# Ensure 'llm_check' column exists
if 'llm_check_fatalities' not in df_classified_articles_relevant.columns:
    df_classified_articles_relevant['llm_check_fatalities'] = None

batch_size = 20  # Save progress every 20 articles
processed_count = 0  # Counter for batch saving

for index, row in df_classified_articles_relevant.iterrows():  # Iterate through the rows of the DataFrame
    if pd.notna(row['llm_check_fatalities']):
        continue  # Skip already processed rows to avoid redoing work

    title = row['title']
    text = row['text']  # Cut the 'text' down to the first 500 characters

    print(title)
    print("\n")

    prompt = create_prompt_check_fatalities(title, text)  # Create the prompt with the truncated text
    response = llm(prompt, stream=True, stop=["\n\n"], temperature=0, max_tokens=20)  # Get response from the LLM

    result = ""
    for output in response:
        result += output['choices'][0]['text']  # Extract and accumulate the text from the response

    result = result.strip()  # Clean up the result

    print("Response:", result)  # Debugging output

    # Store the response in the DataFrame
    deduplicated_df.at[index, 'llm_check_fatalities'] = result

    processed_count += 1

    # Save progress every 20 articles
    if processed_count % batch_size == 0:
        df_classified_articles_relevant.to_csv(output_file, index=False)
        print(f"Saved progress at {processed_count} articles.")

# Final save after loop completion
df_classified_articles_relevant.to_csv(output_file, index=False)
print("Final save completed.")

"Auch für die Tierwelt ist das ein Drama"




llama_perf_context_print:        load time =  229097.94 ms
llama_perf_context_print: prompt eval time =  229097.12 ms /  1721 tokens (  133.12 ms per token,     7.51 tokens per second)
llama_perf_context_print:        eval time =     435.08 ms /     1 runs   (  435.08 ms per token,     2.30 tokens per second)
llama_perf_context_print:       total time =  229534.33 ms /  1722 tokens


Response: 0
Saved progress at 1 articles.
Wann kommt der Wein?




Llama.generate: 102 prefix-match hit, remaining 1626 prompt tokens to eval
llama_perf_context_print:        load time =  229097.94 ms
llama_perf_context_print: prompt eval time =  213644.10 ms /  1626 tokens (  131.39 ms per token,     7.61 tokens per second)
llama_perf_context_print:        eval time =     443.36 ms /     1 runs   (  443.36 ms per token,     2.26 tokens per second)
llama_perf_context_print:       total time =  214090.21 ms /  1627 tokens


Response: 1
Saved progress at 2 articles.
Unwetter: Tote und Verletzte




...

Llama.generate: 102 prefix-match hit, remaining 1565 prompt tokens to eval
llama_perf_context_print:        load time =  229097.94 ms
llama_perf_context_print: prompt eval time =  205622.38 ms /  1565 tokens (  131.39 ms per token,     7.61 tokens per second)
llama_perf_context_print:        eval time =     438.52 ms /     1 runs   (  438.52 ms per token,     2.28 tokens per second)
llama_perf_context_print:       total time =  206063.19 ms /  1566 tokens


Response: 1
Saved progress at 227 articles.
Flutkatastrophe in Süddeutschland: Viertes Todesopfer geborgen




Llama.generate: 102 prefix-match hit, remaining 1453 prompt tokens to eval
llama_perf_context_print:        load time =  229097.94 ms
llama_perf_context_print: prompt eval time =  194081.03 ms /  1453 tokens (  133.57 ms per token,     7.49 tokens per second)
llama_perf_context_print:        eval time =     447.76 ms /     1 runs   (  447.76 ms per token,     2.23 tokens per second)
llama_perf_context_print:       total time =  194531.40 ms /  1454 tokens


Response: 1
Saved progress at 228 articles.
Flut-Chaos in Süddeutschland




Llama.generate: 104 prefix-match hit, remaining 1632 prompt tokens to eval
